**Author(s):** Oliver Baker, Martin Ross  
**Affiliation(s):** Earth and Environmental Sciences, University of Waterloo  
**Date:** 2026-05-06 

---

## Overview
Code used for the Random Forest analysis in submitted manuscript to Quaternary Science Reviews titled 
"Improving regional paleo-ice sheet reconstructions with till provenance analysis: A new approach applied to northern Quebec and Labrador" 

## Data
Will be updated after manuscript acceptance

## Note on AI use
ChapGPT (GPT-5) was used to improve and clean the code, fix some issues, and for annotations



In [ ]:
# Ensure any changes to .csv do not impact future runs 
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# Till data or bedrock data
data = pd.read_csv("TillData_idx.csv")

# Drop rows with missing labels
data_labeled = data.dropna(subset=["Label"])

# Setup training/test split
train_idx, test_idx = train_test_split(data_labeled.index, test_size=0.2, random_state=42)

In [ ]:
# --------------------
# Random Forest Model
# --------------------

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# CLR-transformed vs. untransformed raw data
clr = True
save = False

# N_estimators, class weight, and min_samples_leaf
n = 200
weight = 'balanced_subsample'
leaf = 3

# Till or Bedrock model
model = 'Till'

# Load the dataset
if clr == True: 
    data = pd.read_csv("TillData_CLR.csv")
else:
    data = pd.read_csv("TillData_Untransformed.csv")


# Drop rows with missing labels
# Label can be bedrock type or domain
data_labeled = data.dropna(subset=["Label"])



# Define features and labels

# Using PCA Data
# X = data_labeled[["PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7"]]

# Using non-PCA data
X = data_labeled[['Ni', 'Cu', 'Zn', 'Pb', 'Ba', 'Co', 'Cs', 'Ga', 'Hf', 'Nb', 'Sr', 'Rb', 'Ta', 'Th', 'U', 'V', 'Zr', 'HREE', 'LREE' ]]
y = data_labeled["Label"]

# Split the data into training and testing sets
X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

# Initialize and train the Random Forest classifier

rf = RandomForestClassifier(n_estimators=n,  class_weight=weight, min_samples_leaf=leaf, random_state=42)
rf.fit(X_train, y_train)

# Evaluate the model
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred, labels=rf.classes_)
class_report = classification_report(y_test, y_pred, labels=rf.classes_, zero_division=0)

# Cross-validation
cv_scores = cross_val_score(rf, X_train, y_train, cv=5)
cv_mean_score = cv_scores.mean()

# Predict labels and probabilities for all rows

# PCA Data
# data["Predicted_Label"] = rf.predict(data[["PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7"]])
# probs = rf.predict_proba(data[["PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7"]])

# Non-PCA Data
data["Predicted_Label"] = rf.predict(data[['Ni', 'Cu', 'Zn', 'Pb', 'Ba', 'Co', 'Cs', 'Ga', 'Hf', 'Nb', 'Sr', 'Rb', 'Ta', 'Th', 'U', 'V', 'Zr', 'HREE', 'LREE']])
probs = rf.predict_proba(data[['Ni', 'Cu', 'Zn', 'Pb', 'Ba', 'Co', 'Cs', 'Ga', 'Hf', 'Nb', 'Sr', 'Rb', 'Ta', 'Th', 'U', 'V', 'Zr', 'HREE', 'LREE']])
for i, class_label in enumerate(rf.classes_):
    data[class_label] = probs[:, i]

# Save the updated dataset (Manually change name for iteration)
if save:
    data.to_csv(f"RF_Probabilities_{n}_{weight}_{leaf}.csv", index=False)

# # Save the classification results (Manually change name for iteration)
if save:
    with open(f"RF_Classification_{model}Results_{n}_{weight}_{leaf}.txt", "w") as f:
        f.write(f"Test Set Accuracy: {accuracy * 100:.2f}%\n")
        f.write(f"Cross-Validation Mean Score: {cv_mean_score * 100:.2f}%\n\n")
        f.write("Confusion Matrix:\n")
        f.write(pd.DataFrame(conf_matrix, index=rf.classes_, columns=rf.classes_).to_string())
        f.write("\n\nClassification Report:\n")
        f.write(class_report)



In [ ]:
# ---------------------------------------------------------------------
# Apply trained RF model to till samples (If using bedrock model above)
# ---------------------------------------------------------------------

# Load till dataset (make sure these files have the SAME columns as the bedrock file)
if clr:
    till = pd.read_csv("TillData_CLR.csv")
else:
    till = pd.read_csv("TillData_Untransformed.csv")  

# Use the same feature columns as for bedrock
feature_cols = ['Ni', 'Cu', 'Zn', 'Pb', 'Ba', 'Co', 'Cs', 'Ga', 'Hf', 'Nb',
                'Sr', 'Rb', 'Ta', 'Th', 'U', 'V', 'Zr', 'HREE', 'LREE']

till_features = till[feature_cols]

# Predict class labels and probabilities for till samples using the trained RF
till_pred_labels = rf.predict(till_features)
till_probs = rf.predict_proba(till_features)

# Add predictions to the till dataframe
till["Predicted_Label"] = till_pred_labels
for i, class_label in enumerate(rf.classes_):
    till[class_label] = till_probs[:, i]

# Save till probabilities to a new file
if save:
    till.to_csv(f"RF_TillProbabilities_BedrockModel_{n}_{weight}_{leaf}.csv", index=False)

In [ ]:
# ------------------------------------------
# Determine best hyperparameters (using CV)
# ------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score  # not really needed anymore, but harmless

# X_train, X_test, y_train, y_test defined in previous block

# Input range of min_samples_leaf and values for n_estimators
min_leaf_values = range(1, 9)          # 1 to 9
n_estimators_list = [100, 200, 300, 400, 500, 1000]    # different series on the plot

results = {}

# Initialize best parameter variables
best_cv = -1.0
best_params = None

# Define CV strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Run RF on different values
for n_trees in n_estimators_list:
    cv_scores_for_this_n = []

    for min_leaf in min_leaf_values:
        rf = RandomForestClassifier(
            n_estimators=n_trees,
            min_samples_leaf=min_leaf,
            random_state=42,
            class_weight='balanced_subsample'        # Change weight as needed.
        )

        # Cross-validated accuracy on the training data
        scores = cross_val_score(
            rf,
            X_train,
            y_train,
            cv=cv,
            scoring="accuracy"
        )

        mean_cv = np.mean(scores)
        cv_scores_for_this_n.append(mean_cv)

        # Track best hyperparameters based on mean CV accuracy
        if mean_cv > best_cv:
            best_cv = mean_cv
            best_params = {
                "n_estimators": n_trees,
                "min_samples_leaf": min_leaf
            }

    results[n_trees] = cv_scores_for_this_n

print("Best mean CV accuracy:", best_cv)
print("Best params:", best_params)

# Plotting
plt.figure(figsize=(8, 5))
for n_trees in n_estimators_list:
    plt.plot(
        list(min_leaf_values),
        results[n_trees],
        marker="o",
        label="n_estimators = " + str(n_trees)
    )

plt.xlabel("min_samples_leaf Value")
plt.ylabel("Mean CV Score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('Hyperparameter_BestCV', dpi=300)
plt.show()


In [ ]:
# -----------------------
# SHAP Analysis of Model
# -----------------------

# NOTE: SHAP version 0.45.1 requires numpy version 1.26.4
# Change labels in model for alternative outputs

import shap
import matplotlib.pyplot as plt
import numpy as np

# ELements (Or PC)
feature_names = ['Ni', 'Cu', 'Zn', 'Pb', 'Ba', 'Co', 'Cs', 'Ga', 'Hf', 'Nb',
                 'Sr', 'Rb', 'Ta', 'Th', 'U', 'V', 'Zr', 'HREE', 'LREE']

# Compute SHAP values
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)

print("Classes:", rf.classes_)

# Make 2x2 subplot figure for 4 labels
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i in range(4):
    plt.sca(axes[i])

    if isinstance(shap_values, list):
        class_shap = shap_values[i]
    else:
        class_shap = shap_values[:, :, i]

    shap.summary_plot(
        class_shap,
        X,
        feature_names=feature_names,
        show=False,
        color_bar=True
    )

    axes[i].set_title(str(rf.classes_[i]), fontsize=12)

    # Keep x-axis label only on bottom-right
    if i == 3:
        axes[i].set_xlabel("SHAP value")
    else:
        axes[i].set_xlabel("")

# Remove all repeated Feature value labels
for ax in fig.axes:
    if ax.get_ylabel() == "Feature value":
        ax.set_ylabel("")

# Put Feature value back only on the last colorbar
colorbar_axes = [ax for ax in fig.axes if ax not in axes]
if len(colorbar_axes) > 0:
    colorbar_axes[-1].set_ylabel("Feature value")

plt.tight_layout()

# Till or Bedrock
plt.savefig("TillShapAnalysis.png", dpi=300, bbox_inches="tight")
plt.show()